# 04_Engineering
5 engineering design problems

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
import time
import importlib.util
from google.colab import drive

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/QSO_Research'

# Load QSO
def load_qso():
    spec   = importlib.util.spec_from_file_location(
                 'qso', f'{BASE}/algorithms/qso.py')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.qso

qso_func = load_qso()

print('='*50)
print('NOTEBOOK 05 — ENGINEERING PROBLEMS')
print('='*50)
print('\nProblems:')
problems = {
    'WBD':  'Welded Beam Design',
    'PVD':  'Pressure Vessel Design',
    'TCSD': 'Tension/Compression Spring',
    'SRD':  'Speed Reducer Design',
    'REB':  'Rolling Element Bearing',
}
for k, v in problems.items():
    print(f'  {k}: {v}')
print('\n✅ Ready')

Mounted at /content/drive
NOTEBOOK 05 — ENGINEERING PROBLEMS

Problems:
  WBD: Welded Beam Design
  PVD: Pressure Vessel Design
  TCSD: Tension/Compression Spring
  SRD: Speed Reducer Design
  REB: Rolling Element Bearing

✅ Ready


In [ ]:
# ═══════════════════════════════════════════════════════
# PASTE ALL COMPETITOR FUNCTIONS FROM NOTEBOOK 02 HERE
# Same as Notebook 04 Cell 2b
# ═══════════════════════════════════════════════════════

def initialise_population(pop_size, dim, lb, ub, seed=42):
    """Standardised population initialisation for all algorithms."""
    np.random.seed(seed)
    lb = np.full(dim, lb) if np.isscalar(lb) else np.array(lb)
    ub = np.full(dim, ub) if np.isscalar(ub) else np.array(ub)
    X  = np.random.uniform(lb, ub, (pop_size, dim))
    return X, lb, ub


def evaluate_population(func, X):
    """Evaluate fitness for all agents."""
    return np.array([func(X[i]) for i in range(len(X))])


def bound_check(X, lb, ub):
    """Reflect positions back into bounds."""
    return np.clip(X, lb, ub)


def get_best(fitness, X):
    """Return best fitness and position."""
    idx = np.argmin(fitness)
    return fitness[idx], X[idx].copy()


# Standard return format for ALL algorithms:
# (best_fitness, best_position, convergence_curve)
# This uniform interface is critical for fair comparison

print('✅ Shared utilities defined')

def pso(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        w=0.7, c1=1.5, c2=1.5,
        seed=42):
    """
    Particle Swarm Optimisation (Kennedy & Eberhart, 1995)

    Parameters:
    -----------
    w  : float — inertia weight (default 0.7)
    c1 : float — cognitive coefficient (default 1.5)
    c2 : float — social coefficient (default 1.5)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)

    # Velocities
    V       = np.zeros((pop_size, dim))
    v_max   = 0.2 * (ub - lb)

    # Personal and global bests
    pbest_X = X.copy()
    fitness = evaluate_population(func, X)
    pbest_f = fitness.copy()

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        r1 = np.random.rand(pop_size, dim)
        r2 = np.random.rand(pop_size, dim)

        # Velocity update
        V = (w * V
             + c1 * r1 * (pbest_X - X)
             + c2 * r2 * (gbest_X  - X))
        V = np.clip(V, -v_max, v_max)

        # Position update
        X = X + V
        X = bound_check(X, lb, ub)

        # Fitness evaluation
        fitness = evaluate_population(func, X)

        # Update personal bests
        improved = fitness < pbest_f
        pbest_f[improved] = fitness[improved]
        pbest_X[improved] = X[improved]

        # Update global best
        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
print('Testing PSO...')
from numpy import sum as npsum
sphere = lambda x: npsum(x**2)
f, _, c = pso(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'PSO did not improve'
print('✅ PSO operational')


def ga(func, lb, ub, dim,
       pop_size=30, max_iter=500,
       cr=0.9, mr=0.01,
       seed=42):
    """
    Genetic Algorithm (Holland, 1992)

    Parameters:
    -----------
    cr : float — crossover rate (default 0.9)
    mr : float — mutation rate (default 0.01)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        new_X = np.zeros_like(X)

        for i in range(pop_size):
            # ── Tournament selection ──────────────────────────
            t1, t2 = np.random.randint(0, pop_size, 2)
            parent1 = X[t1] if fitness[t1] < fitness[t2] else X[t2]

            t3, t4 = np.random.randint(0, pop_size, 2)
            parent2 = X[t3] if fitness[t3] < fitness[t4] else X[t4]

            # ── Single-point crossover ────────────────────────
            if np.random.rand() < cr:
                point   = np.random.randint(1, dim)
                child   = np.concatenate([
                              parent1[:point],
                              parent2[point:]])
            else:
                child = parent1.copy()

            # ── Gaussian mutation ─────────────────────────────
            mask          = np.random.rand(dim) < mr
            child[mask]  += np.random.normal(
                                0, 0.1*(ub[mask]-lb[mask]))
            child         = np.clip(child, lb, ub)
            new_X[i]      = child

        X       = new_X
        fitness = evaluate_population(func, X)

        # Elite preservation — keep best from previous generation
        worst_idx = np.argmax(fitness)
        if fitness[worst_idx] > gbest_f:
            X[worst_idx]       = gbest_X.copy()
            fitness[worst_idx] = gbest_f

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
print('Testing GA...')
f, _, c = ga(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'GA did not improve'
print('✅ GA operational')


def de(func, lb, ub, dim,
       pop_size=30, max_iter=500,
       F=0.5, cr=0.9,
       seed=42):
    """
    Differential Evolution (Storn & Price, 1997)
    DE/rand/1/bin variant.

    Parameters:
    -----------
    F  : float — scaling factor (default 0.5)
             Lower values (0.4-0.6) work better on
             continuous unimodal problems. Original
             paper recommends F in [0.4, 1.0].
    cr : float — crossover rate (default 0.9)

    Note on parameters:
    -------------------
    F=0.5 chosen based on parameter sensitivity analysis
    showing F=0.8 causes over-exploration on 30D continuous
    problems with pop_size=30 at 500 iterations.
    This is consistent with Storn & Price (1997) who note
    F in [0.4, 0.6] works well for most continuous problems.
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        for i in range(pop_size):
            # ── Mutation — DE/rand/1 ──────────────────────────
            idxs = list(range(pop_size))
            idxs.remove(i)
            a, b, c_idx = np.random.choice(idxs, 3, replace=False)

            mutant = X[a] + F * (X[b] - X[c_idx])
            mutant = np.clip(mutant, lb, ub)

            # ── Binomial crossover ────────────────────────────
            cross_mask = np.random.rand(dim) < cr
            # Guarantee at least one dimension crosses over
            cross_mask[np.random.randint(dim)] = True
            trial = np.where(cross_mask, mutant, X[i])

            # ── Greedy selection ──────────────────────────────
            trial_f = func(trial)
            if trial_f < fitness[i]:
                X[i]       = trial
                fitness[i] = trial_f
                if trial_f < gbest_f:
                    gbest_f = trial_f
                    gbest_X = trial.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
print('Testing DE (fixed)...')
de_results = []
for seed in [42, 43, 44, 45, 46]:
    f, _, c = de(sphere, -100, 100, dim=30, seed=seed)
    de_results.append(f)
    improvement = (c[0] - f) / c[0] * 100
    print(f'  Seed {seed}: {f:.4e} '
          f'(improvement: {improvement:.1f}%)')

print(f'\n  Mean: {np.mean(de_results):.4e}')
print(f'  Std:  {np.std(de_results):.4e}')

# Convergence check
print('\n  Convergence check (seed 42):')
f, _, c = de(sphere, -100, 100, dim=30, seed=42)
checkpoints = [0, 50, 100, 200, 300, 400, 499]
for cp in checkpoints:
    print(f'    Iter {cp:3d}: {c[cp]:.4e}')

assert f < c[0], 'DE did not improve'
print('\n✅ DE (fixed) operational')


def gwo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Grey Wolf Optimiser (Mirjalili et al., 2014)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    # Alpha, beta, delta wolves
    sorted_idx = np.argsort(fitness)
    alpha_f, alpha_X = fitness[sorted_idx[0]], X[sorted_idx[0]].copy()
    beta_f,  beta_X  = fitness[sorted_idx[1]], X[sorted_idx[1]].copy()
    delta_f, delta_X = fitness[sorted_idx[2]], X[sorted_idx[2]].copy()

    convergence = [alpha_f]

    for t in range(max_iter):
        # Linearly decreasing a from 2 to 0
        a = 2 - 2 * (t / max_iter)

        for i in range(pop_size):
            # Update position based on alpha, beta, delta
            X1 = _gwo_update(X[i], alpha_X, a)
            X2 = _gwo_update(X[i], beta_X,  a)
            X3 = _gwo_update(X[i], delta_X, a)
            X[i] = np.clip((X1 + X2 + X3) / 3, lb, ub)

        fitness = evaluate_population(func, X)

        # Update hierarchy
        sorted_idx = np.argsort(fitness)
        if fitness[sorted_idx[0]] < alpha_f:
            alpha_f = fitness[sorted_idx[0]]
            alpha_X = X[sorted_idx[0]].copy()
        if fitness[sorted_idx[1]] < beta_f:
            beta_f  = fitness[sorted_idx[1]]
            beta_X  = X[sorted_idx[1]].copy()
        if fitness[sorted_idx[2]] < delta_f:
            delta_f = fitness[sorted_idx[2]]
            delta_X = X[sorted_idx[2]].copy()

        convergence.append(alpha_f)

    return alpha_f, alpha_X, convergence


def _gwo_update(x, leader, a):
    """Helper — update position toward a leader wolf."""
    r1, r2 = np.random.rand(len(x)), np.random.rand(len(x))
    A = 2 * a * r1 - a
    C = 2 * r2
    D = np.abs(C * leader - x)
    return leader - A * D


# --- Test ---
print('Testing GWO...')
f, _, c = gwo(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'GWO did not improve'
print('✅ GWO operational')


def woa(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Whale Optimisation Algorithm (Mirjalili & Lewis, 2016)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        a  = 2 - 2 * (t / max_iter)  # Decreases from 2 to 0
        a2 = -1 - (t / max_iter)     # Decreases from -1 to -2

        for i in range(pop_size):
            r  = np.random.rand()
            A  = 2 * a * np.random.rand(dim) - a
            C  = 2 * np.random.rand(dim)
            b  = 1.0   # Spiral shape constant
            l  = (a2 - 1) * np.random.rand() + 1
            p  = np.random.rand()

            if p < 0.5:
                if np.linalg.norm(A) < 1:
                    # Shrinking encircling
                    D       = np.abs(C * gbest_X - X[i])
                    X[i]    = gbest_X - A * D
                else:
                    # Random search
                    rand_X  = X[np.random.randint(pop_size)]
                    D       = np.abs(C * rand_X - X[i])
                    X[i]    = rand_X - A * D
            else:
                # Spiral bubble-net attack
                D       = np.abs(gbest_X - X[i])
                X[i]    = (D * np.exp(b * l)
                           * np.cos(2 * np.pi * l)
                           + gbest_X)

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
print('Testing WOA...')
f, _, c = woa(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'WOA did not improve'
print('✅ WOA operational')


def sca(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Sine Cosine Algorithm (Mirjalili, 2016)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        # Decreasing r1 from 2 to 0
        r1 = 2 - 2 * (t / max_iter)

        for i in range(pop_size):
            r2 = 2 * np.pi * np.random.rand(dim)
            r3 = np.random.rand(dim)
            r4 = np.random.rand()

            if r4 < 0.5:
                X[i] = (X[i]
                        + r1 * np.sin(r2)
                        * np.abs(r3 * gbest_X - X[i]))
            else:
                X[i] = (X[i]
                        + r1 * np.cos(r2)
                        * np.abs(r3 * gbest_X - X[i]))

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
print('Testing SCA...')
f, _, c = sca(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'SCA did not improve'
print('✅ SCA operational')


def hho(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Harris Hawks Optimisation (Heidari et al., 2019)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        E0 = 2 * np.random.rand() - 1   # Initial energy
        E  = 2 * E0 * (1 - t / max_iter) # Escaping energy

        for i in range(pop_size):
            r = np.random.rand()

            if np.abs(E) >= 1:
                # ── Exploration ───────────────────────────────
                if r >= 0.5:
                    rand_X   = X[np.random.randint(pop_size)]
                    X[i]     = (rand_X
                                - np.random.rand()
                                * np.abs(rand_X
                                - 2 * np.random.rand() * X[i]))
                else:
                    X[i]     = ((gbest_X - np.mean(X, axis=0))
                                - np.random.rand()
                                * (lb + np.random.rand() * (ub - lb)))
            else:
                # ── Exploitation ──────────────────────────────
                J        = 2 * (1 - np.random.rand())
                delta_X  = gbest_X - X[i]

                if r >= 0.5 and np.abs(E) >= 0.5:
                    # Soft besiege
                    X[i] = delta_X - E * np.abs(J * gbest_X - X[i])

                elif r >= 0.5 and np.abs(E) < 0.5:
                    # Hard besiege
                    X[i] = gbest_X - E * np.abs(delta_X)

                elif r < 0.5 and np.abs(E) >= 0.5:
                    # Soft besiege with progressive rapid dives
                    Y = gbest_X - E * np.abs(J * gbest_X - X[i])
                    Z = Y + np.random.rand(dim) * _levy_hho(dim)
                    X[i] = (Y if func(Y) < func(Z) else Z)

                else:
                    # Hard besiege with progressive rapid dives
                    Y = gbest_X - E * np.abs(J * gbest_X - np.mean(X, axis=0))
                    Z = Y + np.random.rand(dim) * _levy_hho(dim)
                    X[i] = (Y if func(Y) < func(Z) else Z)

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def _levy_hho(dim, beta=1.5):
    """Lévy flight helper for HHO."""
    from scipy.special import gamma
    sigma = (gamma(1+beta) * np.sin(np.pi*beta/2) /
             (gamma((1+beta)/2) * beta * 2**((beta-1)/2)))**(1/beta)
    u = np.random.normal(0, sigma, dim)
    v = np.random.normal(0, 1, dim)
    return u / (np.abs(v)**(1/beta))


# --- Test ---
print('Testing HHO...')
f, _, c = hho(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'HHO did not improve'
print('✅ HHO operational')


def mpa(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Marine Predators Algorithm (Faramarzi et al., 2020)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)

    # Elite matrix — top predator
    Elite   = np.tile(gbest_X, (pop_size, 1))
    convergence = [gbest_f]
    P       = 0.5
    FADs    = 0.2

    for t in range(max_iter):
        CF = (1 - t/max_iter) ** (2*t/max_iter)

        RL = 0.05 * _levy_mpa(pop_size, dim)
        RB = np.random.randn(pop_size, dim)

        for i in range(pop_size):
            r  = np.random.rand()
            R  = np.random.rand(dim)

            if t < max_iter / 3:
                # Phase 1 — High velocity ratio (prey moves faster)
                stepsize   = RB[i] * (Elite[i] - RB[i] * X[i])
                X[i]      += P * stepsize

            elif t < 2 * max_iter / 3:
                if i < pop_size // 2:
                    # Phase 2a — Unit velocity ratio (Lévy)
                    stepsize = RL[i] * (Elite[i] - RL[i] * X[i])
                    X[i]    += P * stepsize
                else:
                    # Phase 2b — Unit velocity ratio (Brownian)
                    stepsize = RB[i] * (RB[i] * Elite[i] - X[i])
                    X[i]    += P * CF * stepsize
            else:
                # Phase 3 — Low velocity ratio (predator moves faster)
                stepsize   = RL[i] * (RL[i] * Elite[i] - X[i])
                X[i]      += P * CF * stepsize

            # FADs effect
            if np.random.rand() < FADs:
                U    = np.random.rand(dim) < FADs
                X[i]+= CF * (lb + np.random.rand(dim)*(ub-lb)) * U

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        # Update elite matrix
        Elite = np.tile(gbest_X, (pop_size, 1))
        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def _levy_mpa(n, d, beta=1.5):
    """Lévy flight helper for MPA."""
    from scipy.special import gamma
    sigma = (gamma(1+beta) * np.sin(np.pi*beta/2) /
             (gamma((1+beta)/2) * beta * 2**((beta-1)/2)))**(1/beta)
    u = np.random.normal(0, sigma, (n, d))
    v = np.random.normal(0, 1, (n, d))
    return u / (np.abs(v)**(1/beta))


# --- Test ---
print('Testing MPA...')
f, _, c = mpa(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'MPA did not improve'
print('✅ MPA operational')


def bfo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        n_swim=4, n_tumble=4,
        seed=42):
    """
    Bacterial Foraging Optimisation (Passino, 2002)

    Parameters:
    -----------
    n_swim   : int — swim steps per chemotaxis (default 4)
    n_tumble : int — tumble steps (default 4)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    step_size = 0.1 * (ub - lb)
    iters_per_cycle = max(1, max_iter // (n_tumble * n_swim + 1))

    for t in range(max_iter):
        for i in range(pop_size):
            # ── Tumble — random direction ─────────────────────
            delta = np.random.randn(dim)
            delta /= (np.linalg.norm(delta) + 1e-10)

            # ── Swim — move in tumble direction ───────────────
            for s in range(n_swim):
                X_new    = X[i] + step_size * delta
                X_new    = np.clip(X_new, lb, ub)
                f_new    = func(X_new)

                if f_new < fitness[i]:
                    X[i]       = X_new
                    fitness[i] = f_new
                    if f_new < gbest_f:
                        gbest_f = f_new
                        gbest_X = X_new.copy()
                else:
                    break

        # ── Reproduction — top half survives ─────────────────
        if t % iters_per_cycle == 0:
            sorted_idx   = np.argsort(fitness)
            X            = np.vstack([
                               X[sorted_idx[:pop_size//2]],
                               X[sorted_idx[:pop_size//2]]
                           ])
            fitness      = np.concatenate([
                               fitness[sorted_idx[:pop_size//2]],
                               fitness[sorted_idx[:pop_size//2]]
                           ])

        # Decrease step size over time
        step_size *= 0.99

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
print('Testing BFO...')
f, _, c = bfo(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'BFO did not improve'
print('✅ BFO operational')


def qbso(func, lb, ub, dim,
         pop_size=30, max_iter=500,
         qs_threshold=0.5,
         seed=42):
    """
    Quorum Sensing Bacterial Swarm Optimisation (QBSO)
    Based on: Li et al. (2019)

    QS used as enhancement to bacterial swarm —
    NOT as standalone framework (key distinction from QSO)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    step_size = 0.1 * (ub - lb)

    for t in range(max_iter):
        # ── Compute quorum signal ─────────────────────────────
        f_worst = np.max(fitness)
        f_best  = np.min(fitness)
        epsilon = 1e-10

        if f_worst - f_best < epsilon:
            qs_signal = 0.5
        else:
            qs_signal = np.mean(
                (f_worst - fitness) / (f_worst - f_best + epsilon))

        for i in range(pop_size):
            delta = np.random.randn(dim)
            delta /= (np.linalg.norm(delta) + 1e-10)

            if qs_signal >= qs_threshold:
                # QS triggered — move toward global best
                direction = gbest_X - X[i]
                norm      = np.linalg.norm(direction) + 1e-10
                X[i]     += step_size * (direction/norm)
            else:
                # QS not triggered — random walk
                X[i]     += step_size * delta

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        step_size *= 0.995
        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def qbho(func, lb, ub, dim,
         pop_size=30, max_iter=500,
         qs_threshold=0.5,
         seed=42):
    """
    Quorum Sensing Bacterial Horde Optimisation (QBHO)
    Based on: Alzaqebah et al. (2023)

    QS used to identify optimal bacterial positions —
    NOT as standalone framework (key distinction from QSO)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        # ── Quorum detection ──────────────────────────────────
        f_worst   = np.max(fitness)
        f_best    = np.min(fitness)
        epsilon   = 1e-10

        qs_signal = np.mean(
            (f_worst - fitness) / (f_worst - f_best + epsilon + 1e-10))

        # ── Worst position used as reference (per QBHO paper) ─
        worst_idx = np.argmax(fitness)

        for i in range(pop_size):
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)

            if qs_signal >= qs_threshold:
                # Quorum active — avoid worst, move to best
                X[i] = (X[i]
                        + r1 * (gbest_X - X[i])
                        - r2 * (X[worst_idx] - X[i]))
            else:
                # Quorum inactive — standard foraging
                rand_X = X[np.random.randint(pop_size)]
                X[i]   = X[i] + r1 * (rand_X - X[i])

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Tests ---
print('Testing QBSO...')
f, _, c = qbso(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'QBSO did not improve'
print('✅ QBSO operational')

print('Testing QBHO...')
f, _, c = qbho(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'QBHO did not improve'
print('✅ QBHO operational')


def dbo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Dung Beetle Optimisation (Xue & Shen, 2022)

    Four beetle roles:
    - Ball-rollers  : navigate using celestial cues (exploration)
    - Dancers       : reorient when lost (escape local optima)
    - Foragers      : search near best site (exploitation)
    - Brood-stealers: compete for best positions (intensification)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    # Population split into 4 roles
    n_rollers  = pop_size // 4
    n_dancers  = pop_size // 4
    n_foragers = pop_size // 4
    n_thieves  = pop_size - n_rollers - n_dancers - n_foragers

    # Role index boundaries
    r_end = n_rollers
    d_end = n_rollers + n_dancers
    f_end = n_rollers + n_dancers + n_foragers

    for t in range(max_iter):
        R  = 1 - t / max_iter       # Decreasing radius
        CF = (1 - t/max_iter) ** 2  # Convergence factor

        # ── Ball-rolling beetles (exploration) ────────────────────
        for i in range(r_end):
            if np.random.rand() > 0.9:
                # Dancing reorientation
                X[i] = X[i] + np.tan(
                    np.random.rand(dim)) * np.abs(X[i] - gbest_X)
            else:
                # Navigate toward best with decreasing radius
                r1   = np.random.rand(dim)
                X[i] = X[i] + R * r1 * (gbest_X - X[i])
            X[i] = np.clip(X[i], lb, ub)

        # ── Dancing beetles (escape local optima) ─────────────────
        for i in range(r_end, d_end):
            r1   = np.random.rand(dim)
            X[i] = gbest_X + r1 * np.abs(X[i] - gbest_X) * CF
            X[i] = np.clip(X[i], lb, ub)

        # ── Foraging beetles (exploitation) — FIXED ───────────────
        for i in range(d_end, f_end):
            r1   = np.random.rand(dim)
            r2   = np.random.rand(dim)
            # Move toward global best with random perturbation
            X[i] = (X[i]
                    + r1 * (gbest_X - X[i])
                    + r2 * CF * np.random.randn(dim))
            X[i] = np.clip(X[i], lb, ub)

        # ── Brood-stealing beetles (intensification) ──────────────
        for i in range(f_end, pop_size):
            r1   = np.random.rand(dim)
            r2   = np.random.rand(dim)
            # Steal position near global best
            X[i] = (gbest_X
                    + r1 * CF * (X[i] - gbest_X)
                    + r2 * np.random.randn(dim) * R)
            X[i] = np.clip(X[i], lb, ub)

        # ── Evaluate & update best ────────────────────────────────
        fitness = evaluate_population(func, X)

        # Elite preservation
        worst_idx = np.argmax(fitness)
        if fitness[worst_idx] > gbest_f:
            X[worst_idx]       = gbest_X.copy()
            fitness[worst_idx] = gbest_f

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
print('Testing DBO (fixed)...')
dbo_results = []
for seed in [42, 43, 44, 45, 46]:
    f, _, c = dbo(sphere, -100, 100, dim=30, seed=seed)
    dbo_results.append(f)
    improvement = (c[0] - f) / c[0] * 100
    print(f'  Seed {seed}: {f:.4e} '
          f'(improvement: {improvement:.1f}%)')

print(f'\n  Mean: {np.mean(dbo_results):.4e}')
print(f'  Std:  {np.std(dbo_results):.4e}')

# Convergence check
print('\n  Convergence check (seed 42):')
f, _, c = dbo(sphere, -100, 100, dim=30, seed=42)
checkpoints = [0, 50, 100, 200, 300, 400, 499]
for cp in checkpoints:
    print(f'    Iter {cp:3d}: {c[cp]:.4e}')

assert f < c[0], 'DBO did not improve'
assert np.mean(dbo_results) < 1e3, \
    f'DBO mean still too high: {np.mean(dbo_results):.4e}'
print('\n✅ DBO (fixed) operational')



def poa(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Pelican Optimisation Algorithm (Trojovský & Dehghani, 2022)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        for i in range(pop_size):
            # ── Phase 1: Moving toward prey ───────────────────
            # Random prey selection
            prey_idx  = np.random.randint(pop_size)
            prey_X    = X[prey_idx]
            prey_f    = fitness[prey_idx]

            X1 = X[i] + np.random.rand(dim) * (
                prey_X - np.random.randint(1, 3) * X[i])
            X1 = np.clip(X1, lb, ub)
            f1 = func(X1)

            if f1 < fitness[i]:
                X[i]       = X1
                fitness[i] = f1

            # ── Phase 2: Winging on water surface ─────────────
            R    = 0.2 * (1 - t / max_iter)
            X2   = X[i] + R * (2 * np.random.rand(dim) - 1) * X[i]
            X2   = np.clip(X2, lb, ub)
            f2   = func(X2)

            if f2 < fitness[i]:
                X[i]       = X2
                fitness[i] = f2

            if fitness[i] < gbest_f:
                gbest_f = fitness[i]
                gbest_X = X[i].copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def evo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Electric Eel Foraging Optimiser (EVO)
    Based on: Wang et al. (2024)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        a = 2 * (1 - t / max_iter)  # Decreasing factor

        for i in range(pop_size):
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)

            # ── Electric discharge hunting ────────────────────
            if np.random.rand() < 0.5:
                # Discharge toward best
                X[i] = (X[i]
                        + a * r1 * (gbest_X - X[i])
                        + (1-a) * r2 * (
                            X[np.random.randint(pop_size)] - X[i]))
            else:
                # Passive drift with random component
                beta   = np.random.randn(dim)
                X[i]   = (gbest_X
                          + beta * np.abs(gbest_X - X[i]) * (1 - a))

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Tests ---
print('Testing POA...')
f, _, c = poa(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'POA did not improve'
print('✅ POA operational')

print('Testing EVO...')
f, _, c = evo(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'EVO did not improve'
print('✅ EVO operational')








print('✅ Competitor functions defined')

✅ Shared utilities defined
Testing PSO...
  Sphere 30D: 1.1721e-05
✅ PSO operational
Testing GA...
  Sphere 30D: 6.9737e+00
✅ GA operational
Testing DE (fixed)...
  Seed 42: 3.1691e-01 (improvement: 100.0%)
  Seed 43: 4.8467e+01 (improvement: 99.9%)
  Seed 44: 5.5768e-01 (improvement: 100.0%)
  Seed 45: 6.2047e-04 (improvement: 100.0%)
  Seed 46: 6.0036e+01 (improvement: 99.9%)

  Mean: 2.1876e+01
  Std:  2.6687e+01

  Convergence check (seed 42):
    Iter   0: 7.0687e+04
    Iter  50: 3.1347e+03
    Iter 100: 5.9598e+02
    Iter 200: 2.4100e+01
    Iter 300: 2.4555e+00
    Iter 400: 8.7099e-01
    Iter 499: 3.1914e-01

✅ DE (fixed) operational
Testing GWO...
  Sphere 30D: 1.3550e-31
✅ GWO operational
Testing WOA...
  Sphere 30D: 3.7246e-07
✅ WOA operational
Testing SCA...
  Sphere 30D: 5.3338e-12
✅ SCA operational
Testing HHO...
  Sphere 30D: 1.4046e-84
✅ HHO operational
Testing MPA...
  Sphere 30D: 1.4499e-02
✅ MPA operational
Testing BFO...
  Sphere 30D: 4.8163e-01
✅ BFO operational

In [ ]:
def _levy_gjo(n, d, beta=1.5):
    """Lévy flight helper for GJO."""
    from scipy.special import gamma
    sigma = (gamma(1+beta) * np.sin(np.pi*beta/2) /
             (gamma((1+beta)/2) * beta
              * 2**((beta-1)/2)))**(1/beta)
    u = np.random.normal(0, sigma, (n, d))
    v = np.random.normal(0, 1, (n, d))
    return u / (np.abs(v)**(1/beta))


def gjo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Golden Jackal Optimizer (Chopra & Ansari, 2022)
    Published: Expert Systems with Applications, 198, 116924

    Models male and female jackal hunting behaviour:
    - Male jackal: tracks prey (global best)
    - Female jackal: supports male (second best)
    - Prey escape energy decreases over iterations
    """
    X, lb, ub = initialise_population(
                    pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    # Male and female jackal (best two solutions)
    sorted_idx = np.argsort(fitness)
    male_pos   = X[sorted_idx[0]].copy()
    male_f     = fitness[sorted_idx[0]]
    female_pos = X[sorted_idx[1]].copy()
    female_f   = fitness[sorted_idx[1]]

    gbest_f     = male_f
    gbest_X     = male_pos.copy()
    convergence = [gbest_f]

    for t in range(max_iter):
        E1 = 1.5 * (1 - t / max_iter)
        RL = 0.05 * _levy_gjo(pop_size, dim)

        for i in range(pop_size):
            E0 = 2 * np.random.rand() - 1
            E  = E1 * E0

            # Update toward male jackal
            D_male   = np.abs(RL[i] * male_pos - X[i])
            X1       = male_pos - E * D_male

            # Update toward female jackal
            D_female = np.abs(RL[i] * female_pos - X[i])
            X2       = female_pos - E * D_female

            # Average of both updates
            X[i] = np.clip((X1 + X2) / 2, lb, ub)

        fitness = evaluate_population(func, X)

        # Update male and female jackals
        sorted_idx = np.argsort(fitness)

        if fitness[sorted_idx[0]] < male_f:
            male_f   = fitness[sorted_idx[0]]
            male_pos = X[sorted_idx[0]].copy()

        if fitness[sorted_idx[1]] < female_f:
            female_f   = fitness[sorted_idx[1]]
            female_pos = X[sorted_idx[1]].copy()

        if male_f < gbest_f:
            gbest_f = male_f
            gbest_X = male_pos.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
sphere = lambda x: np.sum(x**2)

print('Testing GJO...')
f, _, c = gjo(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'GJO did not improve'
print('✅ GJO operational')

Testing GJO...
  Sphere 30D: 4.7636e-134
✅ GJO operational


In [ ]:
# Same registry as Notebook 04 Cell 3
# Copy and paste the full ALGORITHM_REGISTRY dict here

# ── Build unified registry ────────────────────────────────────────
ALGORITHM_REGISTRY = {
    'QSO':  lambda func, lb, ub, dim, seed:
                qso_func(func, lb, ub, dim,
                         pop_size=30, max_iter=500,
                         theta_min=0.3, theta_max=0.7,
                         lambda_=0.05, tau=10, alpha=1.25,
                         seed=seed),
    'PSO':  lambda func, lb, ub, dim, seed:
                pso(func, lb, ub, dim,
                    pop_size=30, max_iter=500,
                    w=0.7, c1=1.5, c2=1.5, seed=seed),
    'GA':   lambda func, lb, ub, dim, seed:
                ga(func, lb, ub, dim,
                   pop_size=30, max_iter=500,
                   cr=0.9, mr=0.01, seed=seed),
    'DE':   lambda func, lb, ub, dim, seed:
                de(func, lb, ub, dim,
                   pop_size=30, max_iter=500,
                   F=0.5, cr=0.9, seed=seed),
    'GWO':  lambda func, lb, ub, dim, seed:
                gwo(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'WOA':  lambda func, lb, ub, dim, seed:
                woa(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'SCA':  lambda func, lb, ub, dim, seed:
                sca(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'HHO':  lambda func, lb, ub, dim, seed:
                hho(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'MPA':  lambda func, lb, ub, dim, seed:
                mpa(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'BFO':  lambda func, lb, ub, dim, seed:
                bfo(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'QBSO': lambda func, lb, ub, dim, seed:
                qbso(func, lb, ub, dim,
                     pop_size=30, max_iter=500, seed=seed),
    'QBHO': lambda func, lb, ub, dim, seed:
                qbho(func, lb, ub, dim,
                     pop_size=30, max_iter=500, seed=seed),
    'DBO':  lambda func, lb, ub, dim, seed:
                dbo(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'POA':  lambda func, lb, ub, dim, seed:
                poa(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'EVO':  lambda func, lb, ub, dim, seed:
                evo(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),

    'GJO': lambda func, lb, ub, dim, seed:
           gjo(func, lb, ub, dim,
               pop_size=30, max_iter=500,
               seed=seed),
}

def run_algorithm(algo_name, func, lb, ub, dim, seed):
    result = ALGORITHM_REGISTRY[algo_name](
                 func, lb, ub, dim, seed)
    return result[0], result[1], result[2]

print('✅ Registry loaded')
print(f'   {len(ALGORITHM_REGISTRY)} algorithms ready')

✅ Registry loaded
   16 algorithms ready


In [ ]:
# ── Build unified registry ────────────────────────────────────────
ALGORITHM_REGISTRY = {
    'QSO':  lambda func, lb, ub, dim, seed:
                qso_func(func, lb, ub, dim,
                         pop_size=30, max_iter=500,
                         theta_min=0.3, theta_max=0.7,
                         lambda_=0.05, tau=10, alpha=1.25,
                         seed=seed),
    'PSO':  lambda func, lb, ub, dim, seed:
                pso(func, lb, ub, dim,
                    pop_size=30, max_iter=500,
                    w=0.7, c1=1.5, c2=1.5, seed=seed),
    'GA':   lambda func, lb, ub, dim, seed:
                ga(func, lb, ub, dim,
                   pop_size=30, max_iter=500,
                   cr=0.9, mr=0.01, seed=seed),
    'DE':   lambda func, lb, ub, dim, seed:
                de(func, lb, ub, dim,
                   pop_size=30, max_iter=500,
                   F=0.5, cr=0.9, seed=seed),
    'GWO':  lambda func, lb, ub, dim, seed:
                gwo(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'WOA':  lambda func, lb, ub, dim, seed:
                woa(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'SCA':  lambda func, lb, ub, dim, seed:
                sca(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'HHO':  lambda func, lb, ub, dim, seed:
                hho(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'MPA':  lambda func, lb, ub, dim, seed:
                mpa(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'BFO':  lambda func, lb, ub, dim, seed:
                bfo(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'QBSO': lambda func, lb, ub, dim, seed:
                qbso(func, lb, ub, dim,
                     pop_size=30, max_iter=500, seed=seed),
    'QBHO': lambda func, lb, ub, dim, seed:
                qbho(func, lb, ub, dim,
                     pop_size=30, max_iter=500, seed=seed),
    'DBO':  lambda func, lb, ub, dim, seed:
                dbo(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'POA':  lambda func, lb, ub, dim, seed:
                poa(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'EVO':  lambda func, lb, ub, dim, seed:
                evo(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),

    'GJO': lambda func, lb, ub, dim, seed:
           gjo(func, lb, ub, dim,
               pop_size=30, max_iter=500,
               seed=seed),
}

def run_algorithm(algo_name, func, lb, ub, dim, seed):
    """Unified runner — returns (best_f, best_x, convergence)."""
    result = ALGORITHM_REGISTRY[algo_name](
                 func, lb, ub, dim, seed)
    return result[0], result[1], result[2]

# ── Quick verification ────────────────────────────────────────────
print('Verifying all algorithms load correctly...')
sphere = lambda x: np.sum(x**2)
errors = []
for algo in ALGORITHM_REGISTRY:
    try:
        f, _, _ = run_algorithm(algo, sphere, -100, 100,
                                dim=10, seed=42)
        print(f'  ✅ {algo:<6}: {f:.4e}')
    except Exception as e:
        print(f'  ❌ {algo:<6}: {e}')
        errors.append(algo)

if errors:
    print(f'\n⚠️  Fix these before proceeding: {errors}')
else:
    print(f'\n✅ All {len(ALGORITHM_REGISTRY)} algorithms ready')

Verifying all algorithms load correctly...
  ✅ QSO   : 2.7269e-07
  ✅ PSO   : 4.0302e-26
  ✅ GA    : 2.4068e-01
  ✅ DE    : 3.3529e-26
  ✅ GWO   : 1.6789e-64
  ✅ WOA   : 3.3734e-19
  ✅ SCA   : 2.0750e-30
  ✅ HHO   : 3.8324e-89
  ✅ MPA   : 2.6528e-09
  ✅ BFO   : 1.0164e-02
  ✅ QBSO  : 7.3820e-02
  ✅ QBHO  : 3.3231e+02
  ✅ DBO   : 2.9809e-06
  ✅ POA   : 1.1152e-140
  ✅ EVO   : 3.5217e-03
  ✅ GJO   : 4.6244e-129

✅ All 16 algorithms ready


In [2]:
def make_engineering_problems():
    """
    Five constrained engineering design problems.
    Standard benchmark problems used in metaheuristic
    literature for real-world validation.

    Constraint handling: penalty function method.
    Infeasible solutions penalised by adding large
    value proportional to constraint violation.
    """
    problems = {}

    # ── Problem 1: Welded Beam Design (WBD) ──────────────────
    def wbd(x):
        h, l, t, b = x
        P   = 6000
        L   = 14
        E   = 30e6
        G   = 12e6
        t_m = 13600
        s_m = 30000
        d_m = 0.25

        M   = P * (L + l/2)
        R   = np.sqrt(l**2/4 + ((h+t)/2)**2)
        J   = 2*(np.sqrt(2)*h*l
                 *(l**2/12 + ((h+t)/2)**2))

        t1  = P / (np.sqrt(2)*h*l)
        t2  = M * R / J
        tau = np.sqrt(t1**2
                      + 2*t1*t2*l/(2*R)
                      + t2**2)

        sigma = 6*P*L / (b*t**2)
        delta = 6*P*L**3 / (E*b*t**3)
        Pc    = (4.013*E
                 * np.sqrt(t**2*b**6/36)
                 / L**2
                 * (1 - t/(2*L)
                    * np.sqrt(E/(4*G))))

        f = 1.10471*h**2*l + 0.04811*t*b*(14+l)

        g1 = tau   - t_m
        g2 = sigma - s_m
        g3 = h - b
        g4 = 0.10471*h**2 + 0.04811*t*b*(14+l) - 5.0
        g5 = 0.125 - h
        g6 = delta - d_m
        g7 = P - Pc

        penalty = 1e6 * np.sum(
            np.maximum(0, [g1,g2,g3,g4,g5,g6,g7])**2)

        return f + penalty

    problems['WBD'] = {
        'func':   wbd,
        'lb':     np.array([0.1, 0.1, 0.1, 0.1]),
        'ub':     np.array([2.0, 10.0, 10.0, 2.0]),
        'dim':    4,
        'optimum': 1.7248,
        'desc':   'Welded Beam Design',
        'vars':   ['h', 'l', 't', 'b'],
        'n_cons': 7,
    }

    # ── Problem 2: Pressure Vessel Design (PVD) ───────────────
    def pvd(x):
        Ts, Th, R, L = x

        f = (0.6224*Ts*R*L
             + 1.7781*Th*R**2
             + 3.1661*Ts**2*L
             + 19.84*Ts**2*R)

        g1 = -Ts + 0.0193*R
        g2 = -Th + 0.00954*R
        g3 = (-np.pi*R**2*L
              - (4/3)*np.pi*R**3
              + 1296000)
        g4 = L - 240

        penalty = 1e6 * np.sum(
            np.maximum(0, [g1,g2,g3,g4])**2)

        return f + penalty

    problems['PVD'] = {
        'func':   pvd,
        'lb':     np.array([0.0625, 0.0625,
                             10.0, 10.0]),
        'ub':     np.array([6.1875, 6.1875,
                             200.0, 200.0]),
        'dim':    4,
        'optimum': 5804.45,
        'desc':   'Pressure Vessel Design',
        'vars':   ['Ts', 'Th', 'R', 'L'],
        'n_cons': 4,
    }

    # ── Problem 3: Tension/Compression Spring (TCSD) ──────────
    def tcsd(x):
        d, D, N = x

        f = (N + 2) * D * d**2

        g1 = 1 - D**3*N / (71785*d**4)
        g2 = ((4*D**2 - d*D)
              / (12566*(D*d**3 - d**4))
              + 1/(5108*d**2) - 1)
        g3 = 1 - 140.45*d / (D**2*N)
        g4 = (D + d)/1.5 - 1

        penalty = 1e6 * np.sum(
            np.maximum(0, [g1,g2,g3,g4])**2)

        return f + penalty

    problems['TCSD'] = {
        'func':   tcsd,
        'lb':     np.array([0.05, 0.25, 2.0]),
        'ub':     np.array([2.00, 1.30, 15.0]),
        'dim':    3,
        'optimum': 0.012665,
        'desc':   'Tension/Compression Spring Design',
        'vars':   ['d', 'D', 'N'],
        'n_cons': 4,
    }

    # ── Problem 4: Speed Reducer Design (SRD) ─────────────────
    def srd(x):
        b, m, z, l1, l2, d1, d2 = x

        f = (0.7854*b*m**2
             * (3.3333*z**2 + 14.9334*z - 43.0934)
             - 1.508*b*(d1**2 + d2**2)
             + 7.4777*(d1**3 + d2**3)
             + 0.7854*(l1*d1**2 + l2*d2**2))

        g1  = 27/(b*m**2*z) - 1
        g2  = 397.5/(b*m**2*z**2) - 1
        g3  = 1.93*l1**3/(m*z*d1**4) - 1
        g4  = 1.93*l2**3/(m*z*d2**4) - 1
        g5  = (np.sqrt((745*l1/(m*z))**2+16.9e6)
               / (110*d1**3) - 1)
        g6  = (np.sqrt((745*l2/(m*z))**2+157.5e6)
               / (85*d2**3) - 1)
        g7  = m*z/40 - 1
        g8  = 5*m/b - 1
        g9  = b/(12*m) - 1
        g10 = 1.5*d1/l1 - 1
        g11 = 1.1*d2/l2 - 1

        penalty = 1e6 * np.sum(np.maximum(0, [
            g1,g2,g3,g4,g5,g6,
            g7,g8,g9,g10,g11])**2)

        return f + penalty

    problems['SRD'] = {
        'func':   srd,
        'lb':     np.array([2.6, 0.7, 17,
                             7.3, 7.3, 2.9, 5.0]),
        'ub':     np.array([3.6, 0.8, 28,
                             8.3, 8.3, 3.9, 5.5]),
        'dim':    7,
        'optimum': 2994.47,
        'desc':   'Speed Reducer Design',
        'vars':   ['b','m','z','l1','l2','d1','d2'],
        'n_cons': 11,
    }

    # ── Problem 5: Rolling Element Bearing (REB) ──────────────
    def reb(x):
        Dm, Db, Z, fi, fo, Pd, l, gamma, phi, K = x

        epsilon = 1e-10

        # Clip to valid ranges to prevent power errors
        gamma = np.clip(gamma, epsilon, 1 - epsilon)
        fi    = np.clip(fi, 0.515 + epsilon, 0.6)
        fo    = np.clip(fo, 0.515 + epsilon, 0.6)
        Db    = max(Db, epsilon)
        Z     = max(Z, 1.0)

        try:
            term1 = (1 - gamma) / (1 + gamma + epsilon)
            term2 = ((fi * (2*fo - 1))
                     / (fo * (2*fi - 1) + epsilon))

            # Guard negative bases before power
            t1 = max(epsilon, term1)
            t2 = max(epsilon, term2)

            fc = (37.91 * (1 + (1.04
                  * t1**1.72
                  * t2**0.41)**10/3)**0.3)

            Z_floor = max(1, int(np.floor(Z)))
            Q = (fc * Db**1.4
                 * np.cos(phi)
                 * Z_floor**0.7 / 2)

            # Maximisation — return negative
            f = -Q

        except Exception:
            f = 0.0  # Infeasible fallback

        # Constraints
        g1 = Pd - (Dm - Db)*np.cos(phi) + 0.5*Db
        g2 = ((Dm + Db)*np.cos(phi)
              - 0.5*Db - Pd)
        g3 = l / (Db + epsilon) - 1.5
        g4 = 0.5 - l / (Db + epsilon)
        g5 = fi - 0.515
        g6 = 0.6 - fi
        g7 = fo - 0.515
        g8 = 0.6 - fo
        g9 = (Z - np.floor(
            np.pi * (Dm - Db)
            / (2 * Db + epsilon)))

        penalty = 1e6 * np.sum(
            np.maximum(0, [
                g1,g2,g3,g4,g5,
                g6,g7,g8,g9])**2)

        return f + penalty

    problems['REB'] = {
        'func':   reb,
        'lb':     np.array([125, 21.875, 11,
                             0.515, 0.515,
                             0, 0.4, 0.02,
                             0.6, 0.02]),
        'ub':     np.array([150, 25, 17,
                             0.6, 0.6,
                             0.1, 0.5, 0.1,
                             0.8, 0.1]),
        'dim':    10,
        'optimum': None,
        'desc':   'Rolling Element Bearing',
        'vars':   ['Dm','Db','Z','fi','fo',
                   'Pd','l','gamma','phi','K'],
        'n_cons': 9,
    }

    return problems


ENG_PROBLEMS = make_engineering_problems()
print('✅ Engineering problems loaded (with REB fix)')
print(f'\n  {"Problem":<6} {"Description":<35} '
      f'{"Vars":<6} {"Cons":<6} {"Optimum"}')
print('  ' + '-'*65)
for name, prob in ENG_PROBLEMS.items():
    opt = (f'{prob["optimum"]:.4f}'
           if prob['optimum'] else 'N/A (max)')
    print(f'  {name:<6} {prob["desc"]:<35} '
          f'{prob["dim"]:<6} {prob["n_cons"]:<6} {opt}')

✅ Engineering problems loaded (with REB fix)

  Problem Description                         Vars   Cons   Optimum
  -----------------------------------------------------------------
  WBD    Welded Beam Design                  4      7      1.7248
  PVD    Pressure Vessel Design              4      4      5804.4500
  TCSD   Tension/Compression Spring Design   3      4      0.0127
  SRD    Speed Reducer Design                7      11     2994.4700
  REB    Rolling Element Bearing             10     9      N/A (max)


In [3]:
# ── SRD feasibility verification ──────────────────────────────────
import json, numpy as np

res = json.load(open(f'{BASE}/results/raw/engineering/all_results.json'))
r = [x for x in res if x['problem']=='SRD' and x['algo']=='QSO'][0]
i = int(np.argmin(r['runs']))
x = np.array(r['solutions'][i])

def srd_parts(x):
    b, m, z, l1, l2, d1, d2 = x
    f = (0.7854*b*m**2 * (3.3333*z**2 + 14.9334*z - 43.0934)
         - 1.508*b*(d1**2 + d2**2)
         + 7.4777*(d1**3 + d2**3)
         + 0.7854*(l1*d1**2 + l2*d2**2))
    g = [27/(b*m**2*z) - 1,
         397.5/(b*m**2*z**2) - 1,
         1.93*l1**3/(m*z*d1**4) - 1,
         1.93*l2**3/(m*z*d2**4) - 1,
         np.sqrt((745*l1/(m*z))**2 + 16.9e6)/(110*d1**3) - 1,
         np.sqrt((745*l2/(m*z))**2 + 157.5e6)/(85*d2**3) - 1,
         m*z/40 - 1, 5*m/b - 1, b/(12*m) - 1,
         1.5*d1/l1 - 1, 1.1*d2/l2 - 1]
    return f, np.array(g)

f, g = srd_parts(x)
viol = np.maximum(0, g)
print('decision vector      :', np.round(x, 6))
print('raw objective f(x)   : %.4f' % f)
print('penalised (reported) : %.4f' % ENG_PROBLEMS['SRD']['func'](x))
print('cited optimum        : 2994.47')
print('max violation        : %.3e  (constraint g%d)' % (viol.max(), int(np.argmax(viol))+1))
print('n violated (>1e-6)   :', int((viol > 1e-6).sum()))
print('penalty contribution : %.4f' % (1e6*np.sum(viol**2)))

KeyError: 'solutions'

In [ ]:
def run_engineering_experiments(problems, algo_list,
                                 seeds, save_path):
    """
    Run all engineering problems for all algorithms.
    Uses identical seeds and population settings as
    benchmark experiments for consistency.
    """
    os.makedirs(save_path, exist_ok=True)
    all_results = []

    print('='*55)
    print('ENGINEERING PROBLEMS — Starting')
    print(f'  Problems:   {len(problems)}')
    print(f'  Algorithms: {len(algo_list)}')
    print(f'  Seeds:      {len(seeds)}')
    print(f'  Total runs: '
          f'{len(problems)*len(algo_list)*len(seeds)}')
    print(f'  Estimated:  ~20-30 minutes')
    print('='*55 + '\n')

    start_time = time.time()

    for prob_name, prob_info in problems.items():
        print(f'\n── {prob_name}: {prob_info["desc"]} ──')
        prob_results = []

        for algo in algo_list:
            runs = []
            solutions = []

            for seed in seeds:
                try:
                    # Use problem-specific bounds and dim
                    result = ALGORITHM_REGISTRY[algo](
                        prob_info['func'],
                        prob_info['lb'],
                        prob_info['ub'],
                        prob_info['dim'],
                        seed
                    )
                    best_f   = float(result[0])
                    best_x   = result[1].tolist()
                    runs.append(best_f)
                    solutions.append(best_x)
                except Exception as e:
                    runs.append(float('inf'))
                    solutions.append([])

            result_entry = {
                'problem':   prob_name,
                'algo':      algo,
                'runs':      runs,
                'mean':      float(np.mean(runs)),
                'std':       float(np.std(runs)),
                'best':      float(np.min(runs)),
                'worst':     float(np.max(runs)),
                'best_x':    solutions[np.argmin(runs)],
            }
            prob_results.append(result_entry)
            all_results.append(result_entry)

            print(f'  ✅ {algo:<6}: '
                  f'best={np.min(runs):.4e}, '
                  f'mean={np.mean(runs):.4e}')

        # Save checkpoint per problem
        chk = f'{save_path}/checkpoint_{prob_name}.json'
        with open(chk, 'w') as f:
            json.dump(prob_results, f)
        print(f'  💾 Checkpoint saved')

    # Save all results
    with open(f'{save_path}/all_results.json', 'w') as f:
        json.dump(all_results, f)

    elapsed = time.time() - start_time
    print(f'\n✅ Engineering experiments complete')
    print(f'   Total time: {elapsed/60:.1f} minutes')

    return all_results


print('✅ Engineering runner defined')

✅ Engineering runner defined


In [ ]:
SEEDS     = list(range(42, 72))
ALGO_LIST = list(ALGORITHM_REGISTRY.keys())
SAVE_PATH = f'{BASE}/results/raw/engineering'

eng_results = run_engineering_experiments(
    problems  = ENG_PROBLEMS,
    algo_list = ALGO_LIST,
    seeds     = SEEDS,
    save_path = SAVE_PATH
)

print('\n✅ All engineering experiments complete')

ENGINEERING PROBLEMS — Starting
  Problems:   5
  Algorithms: 15
  Seeds:      30
  Total runs: 2250
  Estimated:  ~20-30 minutes


── WBD: Welded Beam Design ──
  ✅ QSO   : best=1.7377e+00, mean=2.3883e+00
  ✅ PSO   : best=1.7249e+00, mean=1.8040e+00
  ✅ GA    : best=1.7711e+00, mean=2.7269e+00
  ✅ DE    : best=1.7249e+00, mean=1.7249e+00
  ✅ GWO   : best=1.7265e+00, mean=1.7319e+00
  ✅ WOA   : best=1.7267e+00, mean=1.7470e+00
  ✅ SCA   : best=1.8028e+00, mean=1.9467e+00
  ✅ HHO   : best=1.7685e+00, mean=2.0154e+00
  ✅ MPA   : best=1.8842e+00, mean=2.2877e+00
  ✅ BFO   : best=1.7258e+00, mean=1.7272e+00
  ✅ QBSO  : best=1.7965e+00, mean=2.9195e+00
  ✅ QBHO  : best=1.7372e+00, mean=1.8378e+00
  ✅ DBO   : best=1.7431e+00, mean=1.9698e+00
  ✅ POA   : best=1.7255e+00, mean=1.7275e+00
  ✅ EVO   : best=1.7251e+00, mean=2.3140e+00
  💾 Checkpoint saved

── PVD: Pressure Vessel Design ──
  ✅ QSO   : best=5.9400e+03, mean=6.4526e+03
  ✅ PSO   : best=5.9076e+03, mean=6.3432e+03
  ✅ GA    : best=

/tmp/ipykernel_685/3209661320.py:202: RuntimeWarning: invalid value encountered in scalar power
  fc = 37.91*(1 + (1.04*((1-gamma)/(1+gamma))**1.72
/tmp/ipykernel_685/3209661320.py:203: RuntimeWarning: invalid value encountered in scalar power
  * (fi*(2*fo-1)/(fo*(2*fi-1)))**0.41)**10/3)**0.3
/tmp/ipykernel_685/3209661320.py:205: RuntimeWarning: invalid value encountered in scalar power
  Q   = fc * Db**1.4 * np.cos(phi) * Z**0.7 / 2


  ✅ HHO   : best=8.3503e+09, mean=8.3503e+09
  ✅ MPA   : best=8.3503e+09, mean=8.3504e+09
  ✅ BFO   : best=8.3503e+09, mean=8.3503e+09
  ✅ QBSO  : best=8.3503e+09, mean=8.3503e+09
  ✅ QBHO  : best=8.3503e+09, mean=8.3503e+09
  ✅ DBO   : best=8.3503e+09, mean=8.3503e+09
  ✅ POA   : best=8.3503e+09, mean=8.3503e+09
  ✅ EVO   : best=8.3503e+09, mean=8.3503e+09
  💾 Checkpoint saved

✅ Engineering experiments complete
   Total time: 33.7 minutes

✅ All engineering experiments complete


In [ ]:
# ── Rerun REB only — all algorithms ──────────────────────────────
import json
import os

BASE      = '/content/drive/MyDrive/QSO_Research'
SAVE_PATH = f'{BASE}/results/raw/engineering'
SEEDS     = list(range(42, 72))
ALGO_LIST = list(ALGORITHM_REGISTRY.keys())

print('Rerunning REB with fixed function...')
print(f'  Algorithms: {len(ALGO_LIST)}')
print(f'  Seeds:      {len(SEEDS)}')
print()

reb_results = []
reb_info    = ENG_PROBLEMS['REB']

for algo in ALGO_LIST:
    runs      = []
    solutions = []

    for seed in SEEDS:
        try:
            result = ALGORITHM_REGISTRY[algo](
                reb_info['func'],
                reb_info['lb'],
                reb_info['ub'],
                reb_info['dim'],
                seed
            )
            runs.append(float(result[0]))
            solutions.append(result[1].tolist())
        except Exception as e:
            runs.append(float('inf'))
            solutions.append([])

    entry = {
        'problem': 'REB',
        'algo':    algo,
        'runs':    runs,
        'mean':    float(np.mean(runs)),
        'std':     float(np.std(runs)),
        'best':    float(np.min(runs)),
        'worst':   float(np.max(runs)),
        'best_x':  solutions[np.argmin(runs)],
    }
    reb_results.append(entry)
    print(f'  ✅ {algo:<6}: '
          f'best={np.min(runs):.4e}, '
          f'mean={np.mean(runs):.4e}')

# ── Save updated checkpoint ───────────────────────────────────────
chk = f'{SAVE_PATH}/checkpoint_REB.json'
with open(chk, 'w') as f:
    json.dump(reb_results, f)

# ── Merge with existing results ───────────────────────────────────
with open(f'{SAVE_PATH}/all_results.json', 'r') as f:
    all_results = json.load(f)

# Remove old REB entries and add new ones
all_results = [r for r in all_results
               if r['problem'] != 'REB']
all_results.extend(reb_results)

with open(f'{SAVE_PATH}/all_results.json', 'w') as f:
    json.dump(all_results, f)

print(f'\n✅ REB rerun complete and saved')
print('   Run Cell 7 again to see updated rankings')

Rerunning REB with fixed function...
  Algorithms: 15
  Seeds:      30

  ✅ QSO   : best=8.3574e+09, mean=8.3966e+09
  ✅ PSO   : best=8.3503e+09, mean=8.3509e+09
  ✅ GA    : best=8.3503e+09, mean=8.3503e+09
  ✅ DE    : best=8.3503e+09, mean=8.3505e+09
  ✅ GWO   : best=8.3503e+09, mean=8.3503e+09
  ✅ WOA   : best=8.3503e+09, mean=8.3509e+09
  ✅ SCA   : best=8.3503e+09, mean=8.3509e+09
  ✅ HHO   : best=8.3503e+09, mean=8.3503e+09
  ✅ MPA   : best=8.3503e+09, mean=8.3504e+09
  ✅ BFO   : best=8.3503e+09, mean=8.3503e+09
  ✅ QBSO  : best=8.3503e+09, mean=8.3503e+09
  ✅ QBHO  : best=8.3503e+09, mean=8.3503e+09
  ✅ DBO   : best=8.3503e+09, mean=8.3503e+09
  ✅ POA   : best=8.3503e+09, mean=8.3503e+09
  ✅ EVO   : best=8.3503e+09, mean=8.3503e+09

✅ REB rerun complete and saved
   Run Cell 7 again to see updated rankings


In [ ]:
# ── Engineering Results Summary ───────────────────────────────────
import json
import numpy as np

BASE      = '/content/drive/MyDrive/QSO_Research'
SAVE_PATH = f'{BASE}/results/raw/engineering'

# ── Load results from Drive ───────────────────────────────────────
with open(f'{SAVE_PATH}/all_results.json', 'r') as f:
    eng_results = json.load(f)

ALGO_LIST = ['QSO','PSO','GA','DE','GWO','WOA','SCA',
             'HHO','MPA','BFO','QBSO','QBHO',
             'DBO','POA','EVO']

PROBLEMS = {
    'WBD':  {'desc': 'Welded Beam Design',
             'optimum': 1.7248},
    'PVD':  {'desc': 'Pressure Vessel Design',
             'optimum': 5804.45},
    'TCSD': {'desc': 'Tension/Compression Spring',
             'optimum': 0.012665},
    'SRD':  {'desc': 'Speed Reducer Design',
             'optimum': 2994.47},
    'REB':  {'desc': 'Rolling Element Bearing',
             'optimum': None},
}

# ── Per problem ranking ───────────────────────────────────────────
qso_ranks = []

for prob_name, prob_info in PROBLEMS.items():
    print(f'\n{"="*60}')
    print(f'{prob_name} — {prob_info["desc"]}')
    print(f'{"="*60}')

    prob_res = [r for r in eng_results
                if r['problem'] == prob_name]

    if not prob_res:
        print('  No results found')
        continue

    prob_res.sort(key=lambda x: x['mean'])

    print(f'\n  {"Rank":<5} {"Algo":<7} '
          f'{"Best":<14} {"Mean":<14} {"Std"}')
    print('  ' + '-'*52)

    qso_rank = None
    for rank, r in enumerate(prob_res, 1):
        if r['algo'] == 'QSO':
            qso_rank = rank
            marker = ' ← QSO'
        else:
            marker = ''
        print(f'  {rank:<5} {r["algo"]:<7} '
              f'{r["best"]:<14.4e} '
              f'{r["mean"]:<14.4e} '
              f'{r["std"]:.4e}{marker}')

    if qso_rank:
        qso_ranks.append(qso_rank)

    # Known optimum comparison
    opt = prob_info['optimum']
    if opt:
        qso_r = next((r for r in prob_res
                      if r['algo'] == 'QSO'), None)
        if qso_r:
            gap = (abs(qso_r['best'] - opt)
                   / abs(opt) * 100)
            print(f'\n  Known optimum:  {opt:.6f}')
            print(f'  QSO best:       {qso_r["best"]:.6f}')
            print(f'  QSO gap:        {gap:.4f}%')

# ── Overall engineering ranking ───────────────────────────────────
print(f'\n{"="*60}')
print('OVERALL ENGINEERING RANKING')
print(f'{"="*60}')

algo_means = {}
for algo in ALGO_LIST:
    algo_res = [r for r in eng_results
                if r['algo'] == algo]
    if algo_res:
        # Normalise each problem to [0,1] then average
        algo_means[algo] = np.mean(
            [r['mean'] for r in algo_res])

ranked = sorted(algo_means.items(),
                key=lambda x: x[1])

print(f'\n  {"Rank":<5} {"Algorithm":<10} {"Mean Score"}')
print('  ' + '-'*30)
for rank, (algo, mean) in enumerate(ranked, 1):
    marker = ' ← QSO' if algo == 'QSO' else ''
    print(f'  {rank:<5} {algo:<10} '
          f'{mean:.4e}{marker}')

# ── QSO summary ───────────────────────────────────────────────────
print(f'\n{"="*60}')
print('QSO ENGINEERING SUMMARY')
print(f'{"="*60}')
if qso_ranks:
    print(f'  Per-problem ranks: {qso_ranks}')
    print(f'  Mean rank:         '
          f'{np.mean(qso_ranks):.1f}')
    print(f'  Best rank:         {min(qso_ranks)}')
    print(f'  Worst rank:        {max(qso_ranks)}')
    print(f'  Top-3 finishes:    '
          f'{sum(1 for r in qso_ranks if r <= 3)}')
    print(f'  Top-5 finishes:    '
          f'{sum(1 for r in qso_ranks if r <= 5)}')


WBD — Welded Beam Design

  Rank  Algo    Best           Mean           Std
  ----------------------------------------------------
  1     DE      1.7249e+00     1.7249e+00     6.5955e-15
  2     BFO     1.7258e+00     1.7272e+00     1.3852e-03
  3     POA     1.7255e+00     1.7275e+00     1.8302e-03
  4     GWO     1.7265e+00     1.7319e+00     6.3225e-03
  5     WOA     1.7267e+00     1.7470e+00     3.3711e-02
  6     PSO     1.7249e+00     1.8040e+00     1.7049e-01
  7     QBHO    1.7372e+00     1.8378e+00     1.5839e-01
  8     SCA     1.8028e+00     1.9467e+00     8.6258e-02
  9     DBO     1.7431e+00     1.9698e+00     1.6129e-01
  10    HHO     1.7685e+00     2.0154e+00     2.7507e-01
  11    MPA     1.8842e+00     2.2877e+00     2.8603e-01
  12    EVO     1.7251e+00     2.3140e+00     5.8077e-01
  13    QSO     1.7377e+00     2.3883e+00     4.7008e-01 ← QSO
  14    GA      1.7711e+00     2.7269e+00     5.9707e-01
  15    QBSO    1.7965e+00     2.9195e+00     8.9213e-01

  Know

In [ ]:
# ── Run GJO on engineering problems ──────────────────────────────
import time

BASE      = '/content/drive/MyDrive/QSO_Research'
SAVE_PATH = f'{BASE}/results/raw/engineering'
SEEDS     = list(range(42, 72))
GJO_ONLY  = ['GJO']

print('Running GJO on engineering problems...')
start = time.time()

reb_info = ENG_PROBLEMS['REB']
gjo_eng_results = []

for prob_name, prob_info in ENG_PROBLEMS.items():
    print(f'\n── {prob_name}: {prob_info["desc"]} ──')
    runs      = []
    solutions = []

    for seed in SEEDS:
        try:
            result = ALGORITHM_REGISTRY['GJO'](
                prob_info['func'],
                prob_info['lb'],
                prob_info['ub'],
                prob_info['dim'],
                seed
            )
            runs.append(float(result[0]))
            solutions.append(result[1].tolist())
        except Exception as e:
            runs.append(float('inf'))
            solutions.append([])

    entry = {
        'problem': prob_name,
        'algo':    'GJO',
        'runs':    runs,
        'mean':    float(np.mean(runs)),
        'std':     float(np.std(runs)),
        'best':    float(np.min(runs)),
        'worst':   float(np.max(runs)),
        'best_x':  solutions[np.argmin(runs)],
    }
    gjo_eng_results.append(entry)
    print(f'  ✅ GJO: best={np.min(runs):.4e}, '
          f'mean={np.mean(runs):.4e}')

# ── Merge with existing results ───────────────────────────────────
with open(f'{SAVE_PATH}/all_results.json', 'r') as f:
    all_results = json.load(f)

# Remove any old GJO entries and add new
all_results = [r for r in all_results
               if r['algo'] != 'GJO']
all_results.extend(gjo_eng_results)

with open(f'{SAVE_PATH}/all_results.json', 'w') as f:
    json.dump(all_results, f)

elapsed = time.time() - start
print(f'\n✅ GJO engineering complete')
print(f'   Time: {elapsed/60:.1f} minutes')
print('   Run Cell 7 to see updated rankings')

Running GJO on engineering problems...

── WBD: Welded Beam Design ──
  ✅ GJO: best=2.2691e+00, mean=3.2805e+00

── PVD: Pressure Vessel Design ──
  ✅ GJO: best=8.6933e+03, mean=1.8033e+04

── TCSD: Tension/Compression Spring Design ──
  ✅ GJO: best=1.2748e-02, mean=1.4235e-02

── SRD: Speed Reducer Design ──
  ✅ GJO: best=3.3292e+03, mean=4.2256e+03

── REB: Rolling Element Bearing ──
  ✅ GJO: best=8.3750e+09, mean=8.4564e+09

✅ GJO engineering complete
   Time: 2.0 minutes
   Run Cell 7 to see updated rankings


In [ ]:
# Run this in any notebook to create Notebook 06
import json

BASE = '/content/drive/MyDrive/QSO_Research'

nb = {
    "nbformat": 4,
    "nbformat_minor": 0,
    "metadata": {
        "kernelspec": {
            "name": "python3",
            "display_name": "Python 3"
        }
    },
    "cells": [{
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "# 06_Analysis_Plots\n",
            "Statistical analysis, Wilcoxon tests, ",
            "Friedman ranking, convergence plots, ",
            "results tables for paper"
        ]
    }]
}

path = f'{BASE}/notebooks/06_Analysis_Plots.ipynb'
with open(path, 'w') as f:
    json.dump(nb, f)

print(f'✅ Created: 06_Analysis_Plots.ipynb')

✅ Created: 06_Analysis_Plots.ipynb


In [ ]:
import numpy as np

reb_lb = np.array([125, 21.875, 11, 0.515, 0.515, 0, 0.4, 0.02, 0.6, 0.02])
reb_ub = np.array([150, 25, 17, 0.6, 0.6, 0.1, 0.5, 0.1, 0.8, 0.1])
reb_dim = 10

def reb_objective_raw(x):
    Dm, Db, Z, fi, fo, Pd, l, gamma, phi, K = x
    Z = max(1, round(Z))
    fc = 37.91 * (1 + (1.04*((1-gamma)/(1+gamma))**1.72 *
                  ((fi*(2*fo-1))/(fo*(2*fi-1)))**0.41)**(10/3))**0.3
    capacity = fc * (Z**(2/3)) * (Db**1.8) if Db > 0 else -1e10

    g = []
    g.append(Pd - 0.5*(Dm - Db) - 0.5)
    g.append(0.5*(Dm + Db) - Pd)
    g.append(0.515 - fi)
    g.append(0.515 - fo)
    g.append(l/Db - 0.4)
    g.append(0.6 - l/Db)
    g.append(K*Pd - 0.5*(Dm - Db))
    g.append(0.5*(Dm + Db) - K*Pd)
    g.append((Dm - Db - 2*Pd) / 2 - (Dm + Db)/2 + Db*0.5)

    violations = [max(0, gi) for gi in g]
    return capacity, violations

def call_algo(algo_fn, func, lb, ub, dim, seed):
    """Wrapper that handles variable-length return tuples."""
    result = algo_fn(func, lb, ub, dim, seed)
    return result[0], result[1]  # best_fit, best_pos — ignore the rest

coeffs = [1e6, 1e4, 1e2, 1e1]
seeds  = [42, 43, 44]  # reduced for speed

print("Penalty Coefficient Diagnostic — REB")
print("="*60)

for coef in coeffs:
    def reb_penalized(x, coef=coef):
        cap, viol = reb_objective_raw(x)
        penalty = coef * sum(v**2 for v in viol)
        return -cap + penalty

    print(f"\nPenalty coefficient = {coef:.0e}")
    for algo_name, algo_fn in [('PSO', pso), ('GWO', gwo)]:
        results = []
        for seed in seeds:
            best_fit, best_pos = call_algo(algo_fn, reb_penalized, reb_lb, reb_ub, reb_dim, seed)
            cap, viol = reb_objective_raw(best_pos)
            max_viol = max(viol) if viol else 0
            results.append((best_fit, cap, max_viol))

        fits  = [r[0] for r in results]
        caps  = [r[1] for r in results]
        viols = [r[2] for r in results]
        print(f"  {algo_name}: mean_fit={np.mean(fits):.4e}, "
              f"std_fit={np.std(fits):.4e}, "
              f"mean_capacity={np.mean(caps):.4e}, "
              f"mean_max_violation={np.mean(viols):.4e}")

Penalty Coefficient Diagnostic — REB

Penalty coefficient = 1e+06
  PSO: mean_fit=1.0770e+10, std_fit=0.0000e+00, mean_capacity=1.3223e+05, mean_max_violation=7.3427e+01
  GWO: mean_fit=1.0770e+10, std_fit=0.0000e+00, mean_capacity=1.3223e+05, mean_max_violation=7.3427e+01

Penalty coefficient = 1e+04
  PSO: mean_fit=1.0757e+08, std_fit=0.0000e+00, mean_capacity=1.3223e+05, mean_max_violation=7.3427e+01
  GWO: mean_fit=1.0757e+08, std_fit=0.0000e+00, mean_capacity=1.3223e+05, mean_max_violation=7.3427e+01

Penalty coefficient = 1e+02
  PSO: mean_fit=9.4484e+05, std_fit=5.5386e+01, mean_capacity=1.3223e+05, mean_max_violation=7.3430e+01
  GWO: mean_fit=9.4481e+05, std_fit=1.5178e+01, mean_capacity=1.3223e+05, mean_max_violation=7.3428e+01

Penalty coefficient = 1e+01
  PSO: mean_fit=-5.5821e+04, std_fit=2.1949e-02, mean_capacity=1.6816e+05, mean_max_violation=7.4990e+01
  GWO: mean_fit=-5.5821e+04, std_fit=0.0000e+00, mean_capacity=1.6816e+05, mean_max_violation=7.4990e+01


In [ ]:
import numpy as np

reb_lb = np.array([125, 21.875, 11, 0.515, 0.515, 0, 0.4, 0.02, 0.6, 0.02])
reb_ub = np.array([150, 25, 17, 0.6, 0.6, 0.1, 0.5, 0.1, 0.8, 0.1])
reb_dim = 10

def reb_objective_exp_fixed_only(x):
    """Original REB with ONLY the **10/3 precedence bug fixed.
    g1/g2 left exactly as in the original notebook function."""
    Dm, Db, Z, fi, fo, Pd, l, gamma, phi, K = x
    epsilon = 1e-10

    gamma = np.clip(gamma, epsilon, 1 - epsilon)
    fi    = np.clip(fi, 0.515 + epsilon, 0.6)
    fo    = np.clip(fo, 0.515 + epsilon, 0.6)
    Db    = max(Db, epsilon)
    Z     = max(Z, 1.0)

    try:
        term1 = (1 - gamma) / (1 + gamma + epsilon)
        term2 = ((fi * (2*fo - 1)) / (fo * (2*fi - 1) + epsilon))
        t1 = max(epsilon, term1)
        t2 = max(epsilon, term2)

        # ONLY CHANGE: parenthesize 10/3 explicitly
        fc = 37.91 * (1 + (1.04 * t1**1.72 * t2**0.41) ** (10/3)) ** 0.3

        Z_floor = max(1, int(np.floor(Z)))
        Q = fc * Db**1.4 * np.cos(phi) * Z_floor**0.7 / 2
        capacity = Q
    except Exception:
        capacity = -1e10

    # ORIGINAL g1/g2 — unchanged from notebook, exactly as written
    g1 = Pd - (Dm - Db)*np.cos(phi) + 0.5*Db
    g2 = (Dm + Db)*np.cos(phi) - 0.5*Db - Pd
    g3 = l / (Db + epsilon) - 1.5
    g4 = 0.5 - l / (Db + epsilon)
    g5 = fi - 0.515
    g6 = 0.6 - fi
    g7 = fo - 0.515
    g8 = 0.6 - fo
    g9 = (Z - np.floor(np.pi * (Dm - Db) / (2 * Db + epsilon)))

    violations = [max(0, gi) for gi in [g1,g2,g3,g4,g5,g6,g7,g8,g9]]
    return capacity, violations

def call_algo(algo_fn, func, lb, ub, dim, seed):
    result = algo_fn(func, lb, ub, dim, seed)
    return result[0], result[1]

coeffs = [1e6, 1e4, 1e2]
seeds  = [42, 43, 44]

print("Diagnostic — Exponent Bug Fixed Only (g1/g2 untouched)")
print("="*65)

for coef in coeffs:
    def reb_penalized(x, coef=coef):
        cap, viol = reb_objective_exp_fixed_only(x)
        penalty = coef * sum(v**2 for v in viol)
        return -cap + penalty

    print(f"\nPenalty coefficient = {coef:.0e}")
    for algo_name, algo_fn in [('PSO', pso), ('GWO', gwo)]:
        results = []
        for seed in seeds:
            best_fit, best_pos = call_algo(algo_fn, reb_penalized, reb_lb, reb_ub, reb_dim, seed)
            cap, viol = reb_objective_exp_fixed_only(best_pos)
            max_viol = max(viol) if viol else 0
            # also show which constraint is the worst offender
            worst_idx = int(np.argmax(viol))
            results.append((best_fit, cap, max_viol, worst_idx))

        fits  = [r[0] for r in results]
        caps  = [r[1] for r in results]
        viols = [r[2] for r in results]
        worst_idxs = [r[3] for r in results]
        print(f"  {algo_name}: mean_fit={np.mean(fits):.4e}, "
              f"mean_capacity={np.mean(caps):.4e}, "
              f"mean_max_violation={np.mean(viols):.4e}, "
              f"worst_constraint_indices={worst_idxs}")

Diagnostic — Exponent Bug Fixed Only (g1/g2 untouched)

Penalty coefficient = 1e+06
  PSO: mean_fit=8.3503e+09, mean_capacity=6.7104e+03, mean_max_violation=9.1291e+01, worst_constraint_indices=[1, 1, 1]
  GWO: mean_fit=8.3503e+09, mean_capacity=6.5077e+03, mean_max_violation=9.1291e+01, worst_constraint_indices=[1, 1, 1]

Penalty coefficient = 1e+04
  PSO: mean_fit=8.3493e+07, mean_capacity=1.0866e+04, mean_max_violation=9.1291e+01, worst_constraint_indices=[1, 1, 1]
  GWO: mean_fit=8.3493e+07, mean_capacity=1.0866e+04, mean_max_violation=9.1291e+01, worst_constraint_indices=[1, 1, 1]

Penalty coefficient = 1e+02
  PSO: mean_fit=8.2417e+05, mean_capacity=1.0866e+04, mean_max_violation=9.1291e+01, worst_constraint_indices=[1, 1, 1]
  GWO: mean_fit=8.2417e+05, mean_capacity=1.0866e+04, mean_max_violation=9.1291e+01, worst_constraint_indices=[1, 1, 1]
